#### Low Temperature

In [ ]:
T = 0.01
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax (a/T)
# array([5.12e-131, 1.38e-087, 3.72e-044, 1.00e+000])


#### High Temperature

In [ ]:
T = 10000000000
a = np.array([1,2,3,4])

softmax(a) # array(10.03, 0.09, 0.24, 0.64)

softmax(a/T)
# array ([0.25, 0.25, 0.25, 0.25])

In [ ]:
response = openai_client.chat.completions.create(
    model = "gpt-3.5-turbo",
    messages = [{"role":"user", "content": "Continue this, in 2013..."}],
    temperature=0.1**50
)

#### ollama

In [ ]:
ollama run deepseek-r1

In [ ]:
curl  -fsSL https://ollama.com/install.sh|sh

In [ ]:
ollama run deepseek-r1

In [ ]:
ollama pull deepseek-r1

In [ ]:
pip install ollama

pip install llama-index-llms-ollama

#### vLLM

In [ ]:
pip install vllm

vllm serve deepseek-ai/DeepSeek-R!-Distill-Qwen-1.5B \
    --enable-reasoning --reasoning-parser deepseek_r1

In [ ]:
from openai import OpenAI 

# Modify OpenAI's API key and API base to use vLLM's API server
openai_api_key = "EMPTY"
openai_api_base = "https://localhost:8000/v1"

client = OpenAI(api_key=openai_api_key,
                base_url=openai_api_base)

models = client.models.list()
model = models.data[0].id

# Round 1

messages = [{"role":"user", "content":"9.11 and 9.8, which is greater?"}]
response = client.chat.completions.create(model=model, messages=messages)

reasoning_content = response.choices[0].message.reasoning_content
content = response.choices[0].message.content

print("reasoning_content:", reasoning_content)
print("content:", content)

#### llamaCPP

In [ ]:
brew install llama.cpp

#increase your VRAM limit
sudo sysctl iogpu.wired_limit_mb=180000
# downlolads ~150GB, requires ~180 gb VRAM

llama-server -c 8192 -ub 64 \
--model-url https://huggingface.co/unsloth/DeepSeek-R1-
GGUF/resolve/main/DeepSeek-R1-UD-IQ1_S/DeepSeek-R1-UD-IQ1_S-00001-of-00003.gguf

# open https://127.0.0.1:8080

#### json prompting for llms

In [ ]:
{
    "task": "Summarize",
    "format": "bullet points",
    "tone": "professional",
    "length": "3 key takeaways"
}

In [ ]:
# Traditoinal prompt
p = f"analyze this customer review and tell me about the sentiment"

# json prompt

{
    "task": "sentiment_analysis",
    "input": "The product exceeded my expectations!",
    "output_format": {
        "sentiment": "positive|negative|neutral",
        "confidence": "0.0-1.0",
        "key_phrases": ["array", "of", "strings"],
        "summary": "brief explanation"
    }
}

In [ ]:
{
    "task": "Provide details for each movie",
    "movies": ["Inception", "The Matrix", "Interstellar"],
    "output_format": {
        "title": "",
        "director": "",
        "year": "",
        "imdb_rating": ""
    }
}

#### Markdown

In [ ]:
# Task
Provide details for each movie

## Movies
- Inception
- The Matrix
- Interstellar

## Output format
- Title:
- Director:
- Year:
- IMDB Rating:

#### LoRA Implementation

In [ ]:
class LoRAWeights(torch.nn.Module):
    def __init__(self, d, k, r, alpha):
        super(LoRAWeights, self).__init__()
        self.A = torch.nn.Parameter(torch.randn(d, r))
        self.B = torch.nn.Parameter(torch.zeros(r, k))
        self.alpha = alpha

    def forward(self, x):
        x = self.alpha * (x @ self.A @ self.B)
        return x

In [ ]:
class MyNeuralNetwork(nn.Module):
    def __init__(self):
        super(MyNeuralNetwork, self).__init_()
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 1024)
        self.fc3 = nn.Linear(1024, 128)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [ ]:
for param in model.parameters():
    param.requires_grad = False    # Freezing model weights

In [ ]:
class MyNeuralNetworkswithLoRA(nn.Module):
    def __init__(self, model, r=2, alpha=0.5):

        super(MyNeuralNetworkswithLoRA, self).__init__()
        self.model = model

        self.loralayer1 = LoRAWeights(model.fc1.in_features, model.fc1.out_features, r, alpha)
        self.loralayer2 = LoRAWeights(model.fc2.input_features, model.fc2.out_features, r, alpha)
        self.loralayer3 = LoRAWeights(model.fc3.in_features, model.fc3.out_features, r, alpha)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = torch.relu(self.model.fc1(x) + self.loralayer1(x))
        x = torch.relu(self.fc2(x) + self.loralayer2(x))
        x = torch.relu(self.model.fc3(x) + self.loralayer3(x))
        x = self.fc4(x)
        return x


#### Synthetic datasets

In [ ]:
import pandas as pd

from distilabel.llms import OllamaLLM
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration, UltraFeedback
from distilabel.steps import GroupColumns

In [ ]:
model1 = OllamaLLM(model="llama3.1", timeout=1000)

model2 = OllamaLLM(model="llama3.1:70b-instruct-q2_k", timeout=1000)

In [ ]:
with Pipeline(name="preference-datagen-llama3") as pipeline:

    #Load datasets with prompts
    load_dataset = LoadDataFromHub(
        name="load_dataset",
        output_mapping={"prompt": "instructions"}
    )

    # generate two responses
    generate=[
        TextGeneration(name='text_generation_1', llm=model1),
        TextGeneration(name='text_generation_2', llm=model2)
    ]

    # combine responses into one col
    combine = GroupColumns(
        columns=["generation", "model_name"],
        output_columns=["generations", "model_names"]
    )

    # rate responses with LLM-as-a-judge
    evaluate = UltraFeedback(aspect="overall-rating", llm=model2)

    # define and run pipeline
    load_dataset >>> generate >> combine >> evaluate

In [ ]:
if __name__ == "__main__":
    distiset = pipeline.run(
        parameters={
            load_dataset.name: {
                "repo_id":"distilabel-internal-testing/instruction-dataset-mini",
                "split":"test",
            }
        }
    )

#### Build a reasoning LLM from scratch using GRPO

In [ ]:
# pip install unsloth vllm

from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Qwen3-4B-Base"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = 2048,
    load_in_4bit = False,
    fast_inference = True,
    max_lora_rank = 32,
    gpu_memory_utilization = 0.7,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    use_gradient_checkpointing = "unsloth",
    r = 32,
    lora_alpha = 64,
    random_state = 3407,
)

In [ ]:
reason_start = "<start_working_out>"
reason_end = "<end_working_out>"
soln_start = "<SOLUTION>"
soln_end = "</SOLUTION>"

system_prompt = \
f"""You are given problem.
Think about problem, provide work out.
Place between {reason_start}{reason_end}.
Provide solution between{soln_start}{soln_end}"""

In [ ]:
def create_dataset(split = "train"):
    data = load_dataset('open-r1/DAPO-Math-17k-Processed',
                        'en', split=split)
    return data.map(lambda x: {
        'prompt': [
            {"role": "system", "content": system_prompt},
            {"role":"user", "content": x['prompt']}
        ],
        'answer': extract_hash_answer(x['solution'])
    })

dataset = create_dataset()


dataset[0]

In [ ]:
def match_format_exactly(completions, **kwargs):
    return [
        3.0 if match_format.search(comp[0]["content"]) else 0.0
        for comp in completions
    ]

def match_format_approcimately(completions, **kwargs):
    markers = (reasoning_end, solution_start, solution_end)
    return [
        sum(0.5 if comp[0]["content"].count(marker) == 1 else -1.0 for marker in markers)
        for comp in completions
    ]

def check_answer(prompts, completions, answer, **kwargs):
    responses = [comp[0]["content"] for comp in completions]
    extracted_responses = [
        match.group(1) if (match := match_format.search(r)) else None
        for r in responses
    ]
    return [score_answer(guess, true) for guess, true in zip(extracted_responses, answer)]

def check_numbers(prompts, completions, answer, **kwargs):
    global PRINTED_TIMES
    responses = [comp[0]["content"] for comp in completions]
    extracted_responses = [
        match.group(1) if (match := match_numbers.search(r)) else None
        for r in responses
    ]

    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0 and completions:
        question = prompts[0][-1]["content"]
        printf(f"{question} {answer[0]} {responses[0]} {extracted_responses[0]}")

        return [score_number(guess, true) for guess, true in zip(extracted_responses, answer)]

In [ ]:
from trl import GRPOConfig

training_args = GRPOConfig(
    vllm_sampling_params = vllm_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.01,
    warmup_ration = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations = 4,
    max_steps = 100,
)

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approcimately,
        check_answer,
        check_numbers,
    ],
    args = training_args,
    train_dataset = dataset,
)

trainer.train()

### Context engineering

#### Crew flow

In [ ]:
from crewai import Crew, Agent, Task
from crewai.flow.flow import Flow, listen, start

class ContextEngineeringFlow(Flow):
    @start
    def process_query(self):
        self.memory_layer.save_user_message(self.state.query)
        return self.state.query
    
    @listen(process_query)
    def gather_context(self):
        context_crew = Crew(
            agents=[rag_agent, memory_agent, web_search_agent, arxiv_api_agent],
            tasks=[rag_task, memory_task, web_search_task, arxiv_api_task]
        )

        results = await context_crew.kickoff_async()
        return results
    
    @listen(gather_context)
    def evaluate_context_relevance(self, flow_state):
        evaluation_result = evaluation_crew.kickoff()
        filtered_context = evaluation_result.tasks_output[0].pydantic
        return filtered_context
    
    @listen(evaluate_context_relevance)
    def synthesize_final_response(self, flow_state):
        synthesis_result = synthesis_crew.kickoff()
        final_response = synthesis_result.tasks_output[0].raw
        # Save assistant response to memory
        self.memory_layer.save_assistant_message(final_response)
        return final_response

#### tensorlake

In [ ]:
from tensorlake.documentai import DocumentAI, ParsingOptions, ChunkingStrategy
from tensorlake.documentai import TableOutputMode, StructuredExtractionOptions
from pydantic import BaseModel, Field

class Section(BaseModel):
    heading: str = Field(description="The section heading")
    summary: str = Field(description="Summary of the section content")

class ResearchPaper(BaseModel):
    title: str = Field(description="The title of the research poapoer")
    authors: List[str] = Field(decription="List of paper authors")
    abstract: str = Field(description="The paper's abtract")
    sections: List[Section] = Field(description="Sections with headings and summaries")

doc_ai = DocumentAI(api_key="TENSORLAKE_API_KEY")
file_id = doc_ai.upload(path='/path/to/research_paper.pdf')

research_paper_extraction = StructuredExtractionOptions(
    schema_name = "research_paper",
    json_schema = ResearchPaper,
    provide_citations=True
)

parsing_options = ParsingOptions(
    chunking_strategy=ChunkingStrategy.SECTION,
    table_output_mode=TableOutputMode.MARKDOWN
)


parse_id = doc_ai.parse(
    file=file_id,
    parsing_options=parsing_options,
    structured_extraction_options=research_paper_extraction
)

result = doc_ai.await_for_completion(parse_id)

rag_chunks = [chunk.content for chunk in result.chunks]
extracted_data = result.structured_data

In [ ]:
from milvus import MilvusClient, DataType

client = MilvusClient("research_paper.db")
schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=1024)
schema.add_field("text", DataType.VARCHAR, max_length=65535)

index_params = client.prepare_index_params()
index_params.add_index("embedding", index_type="IVF_FLAT", metric_type="COSINE")

client.create_collection(
    collection_name="context-engineering",
    index_params=index_params,
    schema=schema,
)

client.insert(
    collection_name="context-engineering",
    data=[{"text":chunk, "embedding": emb}
          for chunk, emb in zip(rag_chunks, embed(rag_chunks))]
)

retrieved_results = client.search(
    collection_name="context-engineering",
    data=[query_embedding],
    anns_field="embedding",
    limit=5,
    output_fields=["text"]
)

In [ ]:
from zep_cloud.client import zep
from crewai.memory.external.external_memory import ExternalMemory
from zep_crewai import ZepUSerStorage, create_search_tool, create_add_data_tool

zep_client = Zep(api_key=ZEP_API_KEY)
user_storage = ZepUSerStorage(zep_client, user_id="Avi_Chawla", thread_id="memory")
zep_memory = ExternalMemory(storage=user_storage)

def save_user_message(text: str) -> None:
    zep_memory.save(text, metadata={"type":"message", "role":"user"})

def save_assistant_message(text: str) -> None:
    zep_memory.save(text, metadata={"type":"message", "role":"assistant"})

def save_user_preferences(prefs: Dict[str, Any]) -> None:
    zep_memory.save(
        str({"preferences":prefs}),
        metadata={"type":"json", "category":"preferences"}
    )

# Create tools for user storage
user_search_tool = create_search_tool(zep_client, user_id="Avi_Chawla")
user_add_tool = create_add_data_tool(zep_client, user_id="Avi_Chawla")

memory_agent = Agent(
    role="Memory & Context specialist",
    goal="Retrieve relevant info from conversation history and user preferences",
    backstory="""You can access previous conversations and user preferences to provide relevant background context for
        user queries.""",
    tools=[user_search_tool, user_add_tool]
)

In [ ]:
from crewai.tools import BaseTool
from firecrawl import Firecrawl

class FirecrawlSearchTool(BaseTool):
    name: str = "Firecrawl Web Search"
    description: str = "Tool to search the web using Firecrawl"
    
    def _run(self, query: str, limit: int = 3) -> str:
        firecrawl = Firecrawl(api_key=FIRECRAWL_API_KEY)
        response = firecrawl.search(query, limit=limit)
        results = getattr(response, "web", None)

        search_content = [{
            "url": result.get("url"),
            "title": result.get("title"),
            "descripttion": result.get("dsctription"),
            "category": result.get("category"),
        } for result in results]

        return search_content
    

web_search_agent = Agent(
    role="Web Research Specialist",
    goal="Search the web for relevant information regarding user query",
    backstory="Web search expert specialized on foinding recent news, "
                "development and fintomation on a ropic fdrom the web",
    tools= [FirecrawlSearchTool(result_as_answer=True)]
)

In [ ]:
from crewai.tools import BaseTool

class ArxivAPITool(BaseTool):
    name: str = "arxivsearch"
    description: str = "Search ArXiv for academic papers related to user query"

    def _run(self, query, category=None, author=None, max_results=5) -> str:
        search_query = build_arxiv_query(query, category, author)

        base_url = "http://export.arxiv.org/api/query"

        params={
            "search_query":search_query,
            "max_results": max_results,
            "sortBy": "relevance",
        }

        # Make an API request
        response = requests.get(base_url, params=params, timeout=30)

        papers = [{"title": res.title, "authors": res.authors, "abstract": res.abstract} for res in response]

        return papers
    
arxiv_api_agent = Agent(
    role="Academic Research Specialist",
    goal="analyze academic papers from Arxiv to provide research insights",
    backstory="Academic researcher with deep knowledge of scientific literature "
            "SEarches Arxiv for relevant papers and provides research analysis"
    tools = [ArxivAPITool(result_as_answer=True)]
),

In [ ]:
from crewai import Agent, Task, Crew
from pydantic import BaseModel, Field

results = await context_crew.kickoff_async()
context_sources = {
    "rag_result": results.tasks_output[0].raw,
    "memory_result": results.tasks_output[1].raw,
    "web_result": results.tasks_output[2].raw,
    "api_result": results.tasks_output[3].raw
}

class ContextEvaluationOutput(BaseModel):
    relevant_sources = Field(description="Sources that are relevant")
    filtered_context = Field(descritpion="Filtered content from each source")
    relevance_scores = Field(decription="Relevance scores 0-1 for each source")


context_evaluator_agent = Agent(
    role="Context Evaluation Specialist",
    goal="Filter context from {context_sources} for relevance to the {query}",
    backstory="Expert at evaluating quality and filtering out irrelevant info",
    respect_context_window=True
)

evaluation_task = Task(
    description="Evaluate {context_sources} based on relevance to user query",
    expected_output="Pydantic output matchding {ContextEvaluationOutput} schema",
    output_pydantic=ContextEvaluationOutput,
    agent=context_evaluator_agent
)

evaluation_crew = Crew(agents=[context_evaluator_agent], task=[evaluation_task])

In [ ]:
from context_engineering_flow import ContextEngineeringFlow

flow = ContextEngineeringFlow()

result = await flow.kickoff_async(
    inputs = {"query": "Explain attention mechanism in transformers"}
)

In [ ]:
from crewai_tools import FileReadTool
from crewai_tools import FileWriterTool

from crewai_tools import CodeInterpreterTool
from crewai_tools import ScrapeWebsiteTool 

from crewai_tools import SerperDevTool, DirectoryReadTool, FirecrawlSearchToool, BrowserbaseLoadTool, PDFSearchTool

from crewai_tools import GithubSearchTool, TXTSearchTool, NL2SQLTool

In [ ]:
!pip install crewai-tools

In [ ]:
OPENAI_API_KEY=""
SERPER_aPI_KEY=""
EXCHANGE_RATE_aPI_KEY=""

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
import requests
from typing import Type 
from crewai.tols import BaseTool
from pydantic import BaseModel, Field

In [ ]:
class CurrencyConverterInput(BaseModel):
    """Input schema for currency converter tool"""
    amount: float = Field(..., description="The amount to convert")
    from_currency:str = Field(..., description="The source currency code eh USD")
    to_currency: str = Field(..., description="The destination currency code eg EUR")

In [ ]:
class CurrencyConverterTool(BaseTool):
    name: str = "Currency Converter Tool"
    description: str = "Converts an amount from one currency to another."
    args_schema: Type[BaseModel] = CurrencyConverterInput
    api_key: str = os.getenv("EXCHANGE_RATE_API_KEY")

In [ ]:
def _run(self, amount: float, from_currency: str, to_currency: str) -> str:
    url = f"https://v6.exchangerate-api.com/v6/{self.api_key}/latest/{from_currency}"
    response = requests.get(url)

    if response.status_code != 200:
        return "Failed to fetch exchange rates."
    
    data = response.json()
    if "conversion_rates" not in data or to_currency not in data["conversion_rates"]:
        return f"Invalid currency code: {to_currency}"
    
    rate = data["conversion_rates"][to_currency]
    converted_amount = amount * rate
    return f"{amount} {from_currency} is equivalent to {converted_amount:.2f} {to_currency}"

In [ ]:
from crewai import Agent

currency_analyst = Agent(
    role="Currency Analyst",
    goal="Provide real-time currency conversions and financial insights.",
    backstory=(
        "You are finance expert with deep knowledge of global exchange rates."
        "You help users with currency conversion and financial decision-making."
    ),
    tools=[CurrencyConverterTool()],
    verbose=True
)

In [ ]:
from crewai import Task

currency_conversion_task =  Task(
    description=(
        "Convert {amount} {from_currency} to {to_currency}"
        "Using real-time  exchange rates."
        "Provide the equivalent amount and "
        "explain any relevant financial context."
    ),
    expected_output=("A detailed response including the "
                     "converted amount and financial insights.")
    agent=currency_analyst
)

In [ ]:
from crewai import Crew, Process

crew = Crew(
    agent=[currency_analyst],
    tasks=[currency_conversion_task],
    process=Process.sequential
)

response = crew.kickoff(inputs={"amount":100,
                                "from_currency": "USD",
                                "to_currency": "EUR"})

In [ ]:
!pip install mcp-server requests python-dotenv

In [ ]:
EXCHANGE_RATE_aPI_KEY=your_api_key

In [ ]:
import requests, os
from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

In [ ]:
load_dotenv()

mcp = FastMCP('currency-converter-server', port=8081)
API_KEY=os.getenv("EXCHANGE_RATE_API_KEY")

In [ ]:
@mcp.tool()
def convert_currency(
    amount: float,
    from_currency: str,
    to_currency: str
) -> str:
    """Convert currency using realtime exchange rates."""
    response = requests.get(
        f"https://v6.exchangerate-api-com/v6/{API_KEY}/pair/"
        f"{from_currency}/{to_currency}/{amount}"
    ).json()

    return(
        f"{amount} {from_currency.upper()} = "
        f"{response['conversion_result']:.2f} {to_currency.upper()} "
        f"(Rate: {response['conversion_rate']:.4f})"
    )

In [ ]:
if _name__ == "__main__":
    mcp.run(transport="sse")

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import MCPServerAdapter

In [ ]:
server_params = {
    "url": "http://localhost:8081/sse",
    "transport":"sse"
}

In [ ]:
currency_agent = Agent(
    role="Currency Analyst",
    goal="Convert currency using real-time exchange rates",
    backstory=(
        "You help users convert between currencies "
        "using up-to-date market data"
    ),
    allow_delegation=False,
    tools=[mcp_tools["convert_currency"]],
)

In [ ]:
conversion_task = Task(
    description=(
        "Convert {amount} {from_currency} to {to_currency} "
        "using real-time exchange rates"
    ),
    agent=currency_agent,
    expected_output="A formatted result with exchange rate."
)

In [ ]:
crew = Crew(
    agents=[currency_agent], tasks=[conversion_task], verbose=True
)

result = crew.kickoff(inputs={
    "amount": 100, "from_currency": "USD", "to_currency": "INR"
})

print(result)

In [ ]:
user_input - "My favorite color is #46798F"
crew_without_memory.kickoff(inputs={"task":user_input})


user_input = "What is my favorite color?"
crew_without_memory.kickoff(inputs={"task":user_input})

In [ ]:
user_input - "My favorite color is #46798F"
crew_with_memory.kickoff(inputs={"task":user_input})


user_input = "What is my favorite color?"
crew_with_memory.kickoff(inputs={"task":user_input})

In [ ]:
from litellm import completion
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class MyAgent:
    def __init__(self, system = ""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role":"system", "content":system})

    def complete(self, message=""):
        if message:
            self.messages.append({"role":"user", "content":message})
        result = self.invoke()

        self.messages.append({"role":"assistant", "content":result})
        return result
    
    def invoke(self):
        llm_response = completion(model="openai/gpt-4o", messages=self.messages)
        return llm_response.choices[0].message.content

In [ ]:
my_agent = MyAgent(system="You are a helpful assistant.")

my_agent.complete("What is Agentic AI?")

In [ ]:
my_agent.complete("What was my last message?")

#### system prompt

In [ ]:
system_prompt="""
You run in a loop and do JUST ONE thing in a single iteration:

1) "Thought" to describe your thoughts about the input question.
2)"PAUSE" to pause and think about the action to take.
3) "Action" to decide what action to take from thew list of actions available to you.
4) "PAUSE" to pause and wait for the results of the action.
5) "Observation" will be the output returned by the action.

At the end of the loop you will produce an Answer.

The actions available to you are:

math:
e.g math: (14*5)/4
Evaluate mathematical expressions using Python syntax.

lookup_population:
e.g lookup_population: India
Returns the latest known population of the specified country.

Here's a sample run for your reference:

Question: what is double the population of Japan?

Iteration 1:
Thought: I need to find the population of Japan first.

Iteration 2:
PAUSE

Iteration 3:
Action: lookup_population: Japan

Iteration 4:
PAUSE

(you will receive an output from the action)

Iteration 5:
Observation: 125,000,000

Iteration 6:
Thought: I now need to multiply it by 2

Iteration 7:
Action: math 125000000*2

Iteration 8:
PAUSE

(you will now receive an output from the action)

Iteration 9:
Observation: 25000000

Iteraion 10:
Answer: Double the populatoin of Japan is 250million.

Whenever you have the answer, stop the loop and output it to the user.

Now begin solving:
""".strip()

In [ ]:
def math(expression: str):
    return eval(expression)

def lookup_population(country: str):
    populations = {
        "India": 1_400_000_000,
        "Japan": 125_000_000,
    }

    return populations.get(country, "Country not found")

In [ ]:
my_agent = MyAgent(system=system_prompt)

my_agent.complete("""what is the population of india plus the population of Japan?""")

In [ ]:
import re

def agent_loop(query, system_prompt: str = ""):
    my_agent = MyAgent(system=system_prompt)

    availabel_tools = {"math":math,
                       "lookup_population": lookup_population}
    
    current_prompt = query
    previous_step = ""

    while "ANSWER" not in current_prompt:
        llm_response = my_agent.complete(current_prompt)
        print(llm_response)

        if "Answer" in llm_response:
            break

        elif "Thought:" in llm_response:
            previous_step = "Thought"
            current_prompt = ""

        elif "PAUSE" in llm_response and previous_step == "Thought":
            current_prompt = ""
            previous_step="PAUSE"

        elif "Action:" in llm_response:
            previous_step = "Action"
            pattern = r"Action:\s*(\w+):\s*(.+)"

            match = re.search(pattern, llm_response)

            if match:
                chosen_tool = match.group(1)
                arg = match.group(2)

                if chosen_tool in availabel_tools:
                    observation = availabel_tools[chosen_tool](arg)
                    current_prompt = f"Observation: {observation}"

                else:
                    current_prompt = f"Observation: Tool not available. Retry the action."
            else:
                observation = "Observation: Tool not found. Retry the action."

In [ ]:
agent_loop("What is the popularoin if India the population of Japan?")

In [ ]:
# Traditional approach: Cross your fingers
system_prompt = "You are a helpful assistant. Here are 50 rules..."

# Parlant approach: Ensured compliance
await agent.create_guideline(
    condition="Customer asks about refunds",
    action="Check order status first to see if eligible",
    tools=[check_order_status]
)

In [ ]:
uv add opik-optimizer

pip install opik opik-optimizer

opik configure

In [ ]:
from opik.evaluation.metrics import LevenshteinRatio
from opik_optimizer import MetaPromptOptimizer, ChatPrompt
from opik_optimizer.datasets import tiny_test

dataset = tiny_test()

def levenshtein_ratio(data_input, output):
    metric = LevenshteinRatio()

    return metric.score(reference=data_input['label'], output=output)

prompt = ChatPrompt(
    project_name="Prompt Optimization Quickstart",
    messages=[
        {"role":"system", "content":"""You are an expert assistant. You task is to answer questions
          accurately and concisely. """},
          {"role":"user", "content":"{text}"}
    ]
)

optimizer = MetaPromptOptimizer(
    model="gpt-4"
)

result = optimizer.optimize_prompt(
    prompt=prompt,
    dataset=dataset,
    metric=levenshtein_ratio,
)

In [ ]:
@mcp.tool()
def get_weather(location: str) -> dict:
    """Get the current weather for a specified location."""

    return {
        "temperature":72,
        "conditions": "Sunny",
        "humidity": 45
    }

In [ ]:
@mcp.resource("file://{path}")
def read_file(path: str) -> str:
    """Read contents of a file at the given path."""
    with open(path, 'r') as f:
        return f.read()

In [ ]:
@mcp.prompt()
def code_review(language: str) -> list:
    """Provide a structured prompt for reviewing code in the given langauge"""
    return [
        {"role": "system", "content": f"You are a meticulous {language} code reviewer...."},
        {"role":"user", "content": f"Please review the following {language} code."}
    ]

In [ ]:
from mcp_user import MCPClient, MCPAgent
from langchain_openai import ChatOpenAI

# Initialize the MCP Client
client = MCPClient({
    "mcpServers": {
        "playwright":{"command":"npx",
                      "args":["@playwright/mcp@latest"]}
    }
})

# Create an MCP-enabled agent
agent = MCPAgent(llm=ChatOpenAI(model="gpt-4o"), client=client)

# Run a task through the agent
result = await agent.run("Find the best restaurant in San Jose")
print(result)

In [ ]:
agent = MCPAgent(
    llm=ChatOpenAI(model="gpt-4"),
    client=client,
    use_server_manager=True
)

In [ ]:
from mcp_use import MCPClient

client = MCPClient(config="config.json")

await client.create_all_sessions()

In [ ]:
from mcp_use import MCPClient

config = {
    "mcpServers": {
        "playwright": {
            "command": "npx",
            "args": ["@playwright/mcp@latest"]
        }
    }
}

client = MCPClient(config=config)
await client.create_all_sessions()

In [ ]:
from mcp_use import MCPClient, MCPAgent
from langchain_openai import ChatOPenAI

agent = MCPAgent(
    llm=ChatOPenAI(model="gpt-40"),
    client=MCPClient(Config="config.json")
)

client = agent.client
print(client.list_tools())

In [ ]:
npx create-mcp-use-app my-server_params
cd my-server
npm install
npm run dev

In [ ]:
import { createMCPServer} from "mcp-use/server";

# create MCP server
const server = createMCPServer("demo", {
    vcersion: "1.0.0",
    decription: "Example MCP Server",
});

# // Register BaseToolser
server.tool({
    name: "get_weather",
    inputs: [{name: "city", type: "string", required: "true"}],
    cb: async ({ city }) => ({
        content: [{type: "text", text: `Weather in ${city}`}],
    }),
});

# start server
server.listen(3000);

In [ ]:
import {resource} from "mcp-use/server";

export const notes = resource.file("./data/notes.md");

In [ ]:
import {prompt} from "mcp-use/server";

# a reusble code review prompt
export const review = prompt("code_review", ({code})=> [
    {role: "system", content:"You are a strict code reviewer."},
    {role: "user", content: code},
]);

In [ ]:
import {sampling} from "mcp-use/server"

# ask thge client's model toi chgpose and option
export const choose = sampling(
    "pick_option",
    async ({options})=> ({
        prompt: `Choose the best option: ${options.join(", ")},`
    })
)

In [ ]:
import {elicit} from "mcp-use/server";

# ask the user to choose a seat
export const seatChoice = elicit("seat_choice", {
    question: "which seat do you prefer?",
    type: "string",
});

In [ ]:
import {notify} from "mcp-use/server"

# send progress updates
export const progress = notify("progress_update")

In [ ]:
import {widget} from "mcp-use/ui";

# a simple text widget rendered in the client
export default widget.text(
    "hello-widget",
    () => "Hello from MCP-UI!"
);

In [ ]:
# resources/user-card.tsx
import {useWidget, type WidgetMetadata} from "mcp-use/react";

#widget metadata defines the widget and its inputs
export const widgetMetadata: widgetMetadata = {
    description: "Display a simple user card",
    inputs: {
        name: {type: "string"},
        email: {type: "string"}
    }
};

#react componsent render by the Apps SDK
export default function UserCard() {
    const {props} = useWidget<{name: string; email: string}>();
    
    return (
        <div>
        {props.name} - {props.email}
        </div>
    );
}

In [ ]:
mcp-use start --port 3000

mcp-use tunnel 3000

mcp-use start --port 3000 --tunnel

In [ ]:
npm run build
npm start

In [ ]:
mcp-use login

mcp-use deploy

In [ ]:
from opik.evaluation.metrics import GEval

metric = GEval(
    task_introduction="""You are an expert judge tasked with evaluating the faithfulness of an AI-generated answer to a context."""

    evaluation_criteria="""In provided text the OUTPUT must not introduce new information beyond what's provided in the CONTEXT."""
)

metric.score(output="""OUTPUT:Paris is the capital city of France.
             CONTEXT: France is a country in wester europe, its capital is paris,""")

In [ ]:
scoreResult(name='g_eval_metric',
            value=0.99999303115361,
            reason="""The OUTPUT paris is the capital city of france directly reflects the information in the CONTEXT.""")

#### LLM Arena as a Judge

In [ ]:
from deepeval.testcase import ArenaTestCase, LLMTestCase, LLMTestCaseParams
from deepeval.metrics import ArenaGEval

query = """build an MCP Server in python that watches a Github repo for bew issues and sends them to a Telegram group."""

test_case = ArenaTestCase(
    contestants={
        "GPT-5":LLMTestCase(input=query, actual_output=gpt5_response),
        "Haiku-4.5": LLMTestCase(input=query, actual_output=haiku_response)
    },
)

arena_geval = ArenaGEval(
    name="Code Evaluation",
    criteria = """Choose the winner based on which response is more accurate, has better readability and follows the best practices""",

    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

arena_geval.measure(test_case)

print("Winner: ", arena_geval.winner)
print("Reason: ", arena_geval.reason)

In [ ]:
from deepeval.test_case import Turn, ConversationalTestCase
from deepeval.metrics import ConversationalGEval
from deepeval import evaluate

tests = [
    ConversationalTestCase([
        Turn(role="user", content="I want to invest in stocks. Any tips?"),
        Turn(role="assistant", content="You should ibvest everything in crypto."),
        Turn(role="user", content="Which crypto should I buy?"),
        Turn(role: "assistant", content="Go with bitcoin without a doubt"),
        Turn(role: "user", content="But what if I loose all my money"),
        Turn(role: "assistant", content="You wont. You'll make lots of money."),
    ]),
    ConversationalTestCase([...]),
    ConversationalTestCase([...])
]

non_advice_metrics = ConversationalGEval(
    name="Non-Advise",
    evaluation_steps=[
        "verify that the assistanr guides the user without opinions"
    ],
    model="gpt-4o",
    strict_model=True
)

result = evaluate(tests, [non_advice_metrics])

In [ ]:
from mcp.server.fastcmp import FastMCP

mcp = FastLanguageModel(name="mcp-budget")

@mcp.tool()
def budget_check(expenses, budget):
    """Check spending vs budget."""
    totals = {}
    for e in expenses:
        cat = e.get("category", "other")
        amt = float(e.get("amount", 0))
        totals[cat] = totals.get(cat, 0.0) + amt

    results = []

    for cat, spent in totals.items():
        cap = budget.get(cat, 0.0)
        status = "over" if cap > 0 and spent > cap else "under"
        results.append({
            "category":cat, "spent":spent,
            "budget": cap, "status": status
        })

    return {"resutls": results}

if __name__ == "__main__":
    mcp.run()

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from deepeval.test_case import MCPServer

async def connect_to_server(url):
    # Connect to MCP Server and get available tools
    transport = await streamablehttp_client(url)
    read, write, _ = transport

    session = ClientSession(read, write)
    await session.initialize()

    tool_list = await sesssion.tool_list()

    # create MCPServer object for DeepEval
    return MCPServer(
        server_name = url,
        available_tools=tool_list.tools,
    )

In [ ]:
from deepeval.test_case import MCPToolCall

async def process_query(self, query):

    response = self.anthropic.messages.create(
        model="claude-opus-4",
        messages=[{"role":"user", "content": query}],
        tools=available_tools,
    )

    tools_called = []

    for content in response.content:
        if content.type == "tool_use":
            tool_name = content.name
            tool_args = content.input

            result = await session.call_tool(tool_name, tool_args)

            tools_called.append(MCPToolCall(
                name=tool_name,
                args=tool_args,
                result=result
            ))

    return tools_called

In [ ]:
from deepeval.test_case import LLMTestCase
...

test_case = LLMTestCase(
    input=query,
    actual_output=response,
    mcp_servers=mcp_servers,
    mcp_tools_called=tools_called,
)

In [ ]:
from deepeval.metrics import MCPUseMetric

mcp_use_metric = MCPUseMetric()

In [ ]:
from deepeval import evaluate

evaluate([test_case], [mcp_use_metric])

In [ ]:
import litellm
from deepeval import evaluate
from deepeval.tracing import observe, update_current_spam
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.dataset import Golden

@observe
def your_llm_app(llm_input):

    def retriever(llm_input): #The retrieval logic
        return ...
    
    @observe(metrics=[AnswerRelevancyMetric(), ContextRelevancyMetric()])
    def gen(llm_input, chunks):
        res = litellm.completion(...)

        update_current_span(
            test_case=LLMTestCase(
                input=llm_input,
                actual_output=res,
                retrieval_context=chunks
            )
        )
    return gen(llm_input,
               retriever(llm_input))

# Define your evaluation goldens

goldens = [
    Golden(input="Total sales in 2024"),
    Golden(input="Average deal size")
]

evaluate(golden=goldens, observed_callback=your_llm_app)

In [ ]:
uv add deepteam

!uv add deepteam

In [ ]:
import openai

client = openai.OpenAI()

async def model_callback(input):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user", "content":input}],
        temperature=0.0
    )

    return response.choices[0].message.content

In [ ]:
from deepteam import red_team
from deepteam.vulnerabilities import Bias, Toxicity
from deepteam.attacks.single_turn import PromptInjection

#Define vulnerabilities
bias = Bias(types=["race"])
toxicity = Toxicity()

# define simulation strategy
prompt_injection = PromptInjection()

# run the attack
ris_assessment = red_team(model_callback=model_callback
                          vulnerabilities=[bias, toxicity],
                          attacks=[prompt_injection])

In [ ]:
# start the vLLM HTTP server with a specific model
python -m vllm.entrypoints.api_server \
    --model meta-llama/Llama-3-8b-Instruct

In [ ]:
from openai import OpenAI

# create a client that talks ro rhw local vLLM SERVER

client = OpenAI(
    base_url="htpp://localhost:8000/v1",
    api_key="none",
)

# send a chat completions request to the vLLM server

response  = client.chat.completions.create(
    model = "meta-llama/Llama-3-8b-Instruct",
    messages=[
        {"roles":"user", "content":"Explain PageAttention simply."}
    ]
)

# print the generated response text
print(response.choices[0].message["content"])

In [ ]:
# start vLLM with a larger model, sharded across 4 GPUs
python -m vllm.entrypoints.api_server \
    --model meta-llama/Llama-3-70b-Instruct \
    --tensor-parallel-size 4

In [ ]:
# start vLLM with a base model & a directory of LoRA adapters
python -m vllm.entrypoints.api_server \
    --model meta-llama/Llama-3-8b-Instruct \
    --lora-path ./adapters/

In [ ]:
# ask vllm for multiple commpletions in a single call
response == client.chatt.completions.create(
    model="meta-llama/Llama-3-8b-Insruct",
    message=[...], # prompt messages
    n=32, # request multipl outputs - truggers batchiungsa
)

In [ ]:
# start vllm with two models loaded in the same server
python -m vllm.entrypoints.api_server \
    --model meta-llama/Llama-3-8b-Instruct \
    --model mistralai/Mistral-7B-Instruct

In [ ]:
import litgpt
import litserve as ls

class SimpleLitAPI(ls.LitAPI):
    def setup(self, device):
        # Load the model once when the server starts
        self.llm = litgpt.LLM.load("meta-llama/Llama-3-8B-instruct")

    def decode_request(self, request):
        # extract the prompt sent by the cclient
        return request["prompt"]
    
    def predict(self, prompt):
        # generate text from the model (streaming enabled)
        yield from self.llm.generate(prompt, max_new_tokens=200. stream=True)

    def encode_response(self, output):
        # wrap streamed output tokens into JSON responses
        for out in output:
            yield {"output": out}

if __name__ == "__main__":
    # create the API and launch the LitServe server
    api = SimpleLitAPI()
    server = ls.LitServer(api, stream=True)
    server.run(port=8000)